In [142]:
!pip install pandas

In [143]:
import pandas as pd
import sqlite3
from sqlite3 import Error

## Подключение к бд

In [150]:
def create_connection(path):
    connection = None
    try:
        connection = sqlite3.connect(path)
    except Error as e:
        print(f"The error {e} occured")
    return connection

connection = create_connection("../checking-logs.sqlite")

## Создание таблицы datamart, удовлетворяющий следующим условиям: 
* таблица должна иметь следующие столбцы: uid, labname, first_commit_ts, first_view_ts
* first_commit_ts - это просто новое имя временной метки столбца из таблицы проверки, оно показывает первую фиксацию из определенной лаборатории и от конкретного пользователя
* first_view_ts - это первое посещение пользователем просмотров страниц таблицы, метка времени, когда пользователь посетил ленту новостей
* статус = «готов» все еще должен быть фильтром
* numTrials = 1 все еще должен быть фильтром
* имена лабораторий все еще должны быть из списка: 'laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1'
* таблица должна содержать только пользователей (uids с user_*), а не администраторов
* first_commit_ts и first_view_ts должны быть разоброены как datetime64[ns]

In [147]:
query = "CREATE TABLE datamart AS " \
"SELECT checker.uid, labname, timestamp AS first_commit_ts, " \
"(SELECT MIN(datetime) FROM pageviews GROUP BY uid HAVING checker.uid = pageviews.uid) AS first_view_ts " \
"FROM checker LEFT JOIN pageviews ON pageviews.uid = checker.uid " \
"WHERE status = 'ready' AND numTrials = 1 AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1') AND checker.uid LIKE 'user_%' " \
"GROUP BY checker.uid, labname, timestamp;"

connection.execute(query)
datamart = pd.read_sql("SELECT * FROM datamart", connection)

datamart["first_commit_ts"] = pd.to_datetime(datamart["first_commit_ts"])
datamart["first_view_ts"] = pd.to_datetime(datamart["first_view_ts"])

datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 4.5+ KB


## Создание датафреймов: test и contol
* test - имеет значения в first_view_ts
* control - отсутствует значения в first_view_ts
* замените недостающие значения в элементе управления средним значением first_view_ts тестовых пользователей
* сохраните обе таблицы в базе данных

In [151]:
test = datamart[datamart["first_view_ts"].notnull()]
control = datamart[datamart["first_view_ts"].isnull()]

In [152]:
control.loc[:, "first_view_ts"] = control["first_view_ts"].fillna(test["first_view_ts"].mean())

In [153]:
control.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.2+ KB


In [109]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 59 entries, 0 to 114
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.3+ KB


In [154]:
test.to_sql(name="test", con=connection)
control.to_sql(name="control", con=connection)

ValueError: Table 'test' already exists.

In [156]:
pd.read_sql("SELECT * FROM control", connection)

,index,uid,labname,first_commit_ts,first_view_ts
0,12,user_11,laba05,2020-05-03 21:06:55.970293,2020-04-27 00:40:05.761783
1,13,user_11,project1,2020-05-03 23:45:33.673409,2020-04-27 00:40:05.761783
2,14,user_12,laba04,2020-04-18 17:07:51.767358,2020-04-27 00:40:05.761783
3,15,user_12,laba04s,2020-04-26 15:42:38.070593,2020-04-27 00:40:05.761783
4,16,user_12,laba05,2020-05-03 08:39:25.174316,2020-04-27 00:40:05.761783
...,...,...,...,...,...
76,135,user_8,laba04s,2020-04-19 10:22:35.761944,2020-04-27 00:40:05.761783
77,136,user_8,laba05,2020-05-02 13:28:07.705193,2020-04-27 00:40:05.761783
78,137,user_8,laba06,2020-05-16 17:56:15.755553,2020-04-27 00:40:05.761783
79,138,user_8,laba06s,2020-05-16 20:01:07.900727,2020-04-27 00:40:05.761783


## Закрытие соединения

In [148]:
connection.close()